# Solving the Install Problem

The **install problem** consists of determining whether a new set of packages can be installed in a system. This application is based on the article [OPIUM: Optimal Package Install/Uninstall Manager](http://cseweb.ucsd.edu/~rjhala/papers/opium.pdf). Many packages depend on other packages to provide some functionality. Each distribution contains a meta-data file that explicates the requirements of each package of the distribution. The meta-data contains details like the name, version, etc. More importantly, it contains **depends** and **conflicts** clauses that stipulate which other packages should be on the system. The depends clauses stipulate which other packages must be present. The conflicts clauses stipulate which other packages must not be present.

Let's see how we can solve this using Z3. The idea is to define a Boolean variable for each package. This variable is true if the package must be in the system. First, let's see a simple package structure example:

In [ ]:
a, b = Bools('a b')
s = Solver()
s.add(Implies(a, b), a)

display_pkg_struct(s)

Here, we have two packages `a` and `b`. The arrow pointing from `a` to `b` represents the dependency from `a` to `b`; if package `a` is to be installed, then package `b` must also be installed. The fact that `a` is shaded blue means that `a` is a requested package and must be installed to satisfy the problem. Let's look at another dependency example:

In [ ]:
a, b, c, d, e, f = Bools('a b c d e f')
s = Solver()
s.add(
    And(
      Implies(a, b),
      Implies(a, c),
      Implies(a, d),
      Implies(d, Or(e, f))),
    a
)

display_pkg_struct(s)

Here, `a` depends on `b`, `c`, and `d`. The joint dependency connnection below `d` means that `d` requires either `e` or `f` to be installed for it to be installed. To make our lives a bit easier, we'll define a helper function that easily lets us define these types of relations:

In [ ]:
def DependsOn(pack, deps):
    if is_expr(deps): # Allows us to do something like DependsOn(a, b) instead of DependsOn(a, [b])
        return Implies(pack, deps)
    else:
        return And([ Implies(pack, dep) for dep in deps ])

In [ ]:
a, b, c, d, e, f = Bools('a b c d e f')

s = Solver()
s.add(
    DependsOn(a, [b, c, d]),
    DependsOn(d, [Or(e, f)]),
    a
)

display_pkg_struct(s)

Up until now, we've just been looking at how to visualize these dependency structures, so now let's look at actually solving them:

In [ ]:
a, b, c, d, e, f = Bools('a b c d e f')

s = Solver()
s.add(
    DependsOn(a, [b, c, d]),
    DependsOn(d, [Or(e, f)]),
    a
)

display_pkg_solution(s)

Through the graph we can easily see a package installation profile that will satisfy our constaints; the packages shaded blue are specifically requested and so must be installed, and the packages shaded green have been chosen by the SAT solver to satisfy the given install problem. In the above example, installing packages `a`, `b`, `c`, `d`, and `f` is enough to satify the constraints. Package `e` is not needed.

Let's look at a slightly more complex example:

In [ ]:
a, b, c, d = Bools('a b c d')

s = Solver()
s.add(
    DependsOn(a, [b, c]),
    DependsOn(c, d),
    Or(Not(b), Not(d)),
    a
)

display_pkg_struct(s)

Here, the red line between `b` and `d` represents a conflict; if package `b` is to be installed, the package `d` must _not_ be installed. Let's try and define a helper function for this. Change the return line in the following function:

In [ ]:
def Conflict(p1, p2):
  return Or( p1, p2 ) # CHANGE THIS LINE

In [ ]:
a, b, c, d = Bools('a b c d')

s = Solver()
s.add(
    DependsOn(a, [b, c]),
    DependsOn(c, d),
    Conflict(b, d),
    a
)

display_pkg_struct(s)

Do you think the above package configuration is satisfiable? Let's see:

In [ ]:
display_pkg_solution(s)

This is because packages `b` and `d` are both necessary _and_ conflicting.

### You're turn!

With these two functions, try and encode the example in the [Opium](http://cseweb.ucsd.edu/~rjhala/papers/opium.pdf) article (Section 2) in Z3. It should look like this when completed:

In [ ]:
display_opium_graph()

In [ ]:
a, b, c, d, e, f, g, y, z = Bools('a b c d e f g y z')

s = Solver()
s.add(
      # INSERT CODE HERE
      a, z)

display_pkg_struct(s)

Then check it here:

In [ ]:
display_pkg_solution(s)

Run the cell below and copy-paste the output string into your homework submission

In [ ]:
pkg_output_string(s)